# 09 · A condition-enriched state with sparse reference coverage

**ScGeo 0.2.0 learning track.** The Enriched state has zero control cells in one simulated sample and one cell in the other. Keep observed expression and abundance visible, but do not extrapolate an unsupported within-state response.

Use the [separate tutorial environment](README.md). Run cells in order in a fresh kernel. Outputs are written to a new ignored results directory; manuscript inputs and outputs are not used.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import scgeo as sg
from IPython.display import display

cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents)
             if (p / "scripts/scgeo_v020_examples/SOURCE_PART1.json").is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook from the scgeo-notebooks checkout.")
examples = root / "scripts/scgeo_v020_examples"
if str(examples) not in sys.path:
    sys.path.insert(0, str(examples))  # only tutorial helpers, not package source
from tutorial_support import setup, save_preview, show_gallery, FORMATS
out = setup(root, "09_sparse_states")

In [ ]:
from condition_sparse_states_gallery import make_example_data, sample_state_counts, STATES, run_gallery
adata = make_example_data(seed=83)
counts = sample_state_counts(adata)
display(counts)
counts.to_csv(out / "sample_state_counts.csv", index=False)
sg.tl.robust_shift(adata, rep="X_pca", condition_key="condition", group0="control",
    group1="treated", by="state", sample_key="sample", center="geometric_median",
    n_boot=0, seed=83)
explanation = sg.tl.explain_shift(adata, modules={"Response":["Ieg1","Response2"]},
    min_cells=10, min_samples=2, n_boot=0, seed=83)
display(explanation["summary"])

## Observed expression without a valid shift axis

A zero-cell sample has no expression value; the singleton has an observed value but no interval. Cell-level IQRs describe the sampled cells rather than treatment-effect uncertainty.

In [ ]:
result = sg.pl.shift_curves(adata, state="Enriched", var_names=["Ieg1"],
    panels=("distribution","expression"), basis="display", return_result=True, show=False)
display(result.data["coverage"])
display(result.data["sample_expression"])
save_preview(result, out, "enriched_expression")

## PAGA pies: two distinct denominators

Use all eligible annotated cells as the denominator population; do not subset to the enriched state first. Raw cell fractions describe capture composition. Sample-abundance sectors normalize equally weighted sample frequencies within conditions. Keep pooled positions separate from observed group endpoints.

In [ ]:
common = dict(node_key="state", condition_key="condition", group0="control",
              group1="treated", basis="display", paga_key="paga_fixture", graph_node_order=STATES,
              node_position="pooled", node_scale_by_n=False,
              show_shift=False, show_endpoints=False, show_velocity=False,
              return_result=True, show=False)
result = sg.pl.paga_shift_map(adata, **common, pie_key="sample", pie_mode="cell_fraction")
save_preview(result, out, "sample_pies")
result = sg.pl.paga_shift_map(adata, **common, pie_key="condition",
                             pie_mode="sample_abundance", pie_sample_key="sample")
display(result.data["pie_sample_counts"])
display(result.data["pie_values"])
save_preview(result, out, "sample_normalized_pies")

## Optional broader population analysis

On a copy, retain the original state annotations and create a separately justified parent grouping. The parent effect can mix changes in the original states' abundance and within-state response; it does not repair missing reference cells.

In [ ]:
parent = adata.copy()
original = parent.obs["state"].astype(str)
parent.obs["parent_state"] = pd.Categorical(original.where(~original.isin(["Shared","Enriched"]), "Parent"))
sg.tl.robust_shift(parent, rep="X_pca", condition_key="condition", group0="control",
    group1="treated", by="parent_state", sample_key="sample", center="geometric_median",
    store_key="parent_shift", n_boot=0, seed=83)
parent_result = sg.tl.explain_shift(parent, shift_key="parent_shift", states=["Parent"],
    local_key="state", store_key="parent_explanation", n_boot=0, seed=83)
display(parent_result["summary"])
result = sg.pl.shift_curves(parent, key="parent_explanation", state="Parent",
    var_names=["Ieg1"], basis="display", color="state", return_result=True, show=False)
save_preview(result, out, "parent_response")

## Validate sampling behavior

The full gallery checks the absent and singleton controls, and verifies that uniform duplication of one mouse's captured cells changes raw-cell pies but not sample-normalized abundance pies.

In [ ]:
benchmark = run_gallery(out / "gallery", formats=FORMATS)
display(benchmark["checks"])
display(benchmark["metrics"])

See [condition-sparse states](https://github.com/liuifrec/scgeo/blob/v0.2.0/docs/CONDITION_SPARSE_STATES.md). The simulated grouping is not a demonstrated biological lineage.